In [31]:
def clone_sparse_checkout(repo_url: str, target_dir: str, folder_path: str, branch: str = "main"):
    """
    Clone only a specific folder from a GitHub repo using sparse checkout.
    """

    import os
    import shutil
    import subprocess
    
    target_dir = os.path.abspath(target_dir)

    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)

    subprocess.run(
        ["git", "clone", "--no-checkout", "--depth", "1", repo_url, target_dir],
        check=True,
    )

    cwd = os.getcwd()
    try:
        os.chdir(target_dir)
        subprocess.run(["git", "sparse-checkout", "init", "--cone"], check=True)
        subprocess.run(["git", "sparse-checkout", "set", folder_path], check=True)
        subprocess.run(["git", "checkout", branch], check=True)
    finally:
        os.chdir(cwd)

In [32]:
import os

this_folder = os.getcwd()
target_dir = os.path.join(this_folder,"doc")
os.makedirs(target_dir,exist_ok=True)

repo_url = "https://github.com/duckdb/duckdb-web.git"
folder_path = "docs/current"
    
    final_collection_path = os.path.join(target_dir,folder_path)

In [33]:
clone_sparse_checkout(repo_url, target_dir,folder_path)

Cloning into '/Users/ss/duckdb-assistant/scripts/doc'...


Your branch is up to date with 'origin/main'.


Already on 'main'


In [34]:

def find_markdown_files(root_folder: str) -> List[Path]:
    """
    Recursively find all .md files under root_folder.
    """
    from pathlib import Path
    root = Path(root_folder)
    return [p for p in root.rglob("*.md") if p.is_file()]

In [35]:
markdown_files = find_markdown_files(final_collection_path)

In [36]:
def extract_title(md_text: str, fallback: str) -> str:
    for line in md_text.splitlines():
        line = line.strip()
        if line.startswith("# "):
            return line[2:].strip()
    return fallback

    

In [37]:
def load_markdown_docs(root_folder: str, repo_name: str) -> List[Dict]:
    """
    Load markdown files and create records with id, title, content, and metadata.
    """
    
    
    docs = []
    md_files = find_markdown_files(root_folder)

    for idx, md_path in enumerate(md_files):
        content = md_path.read_text(encoding="utf-8", errors="ignore")
        rel_path = md_path.as_posix()
        title = extract_title(content, fallback=md_path.stem)

        doc_id = f"{repo_name}:{idx}:{md_path.relative_to(root_folder).as_posix()}"

        docs.append(
            {
                "id": doc_id,
                "title": title,
                "content": content,
                "metadata": {
                    "title": title,
                    "source_path": rel_path,
                    "file_name": md_path.name,
                    "extension": md_path.suffix,
                },
            }
        )
    return docs

In [38]:
docs = load_markdown_docs(final_collection_path, "duckdb-docs")

In [39]:
len(docs)

417

In [40]:
def store_in_chroma(docs: List[Dict], collection_name: str = "markdown_docs", persist_dir: str = "./chroma_db"):
    """
    Store docs in ChromaDB.
    """
    import chromadb
    
    client = chromadb.PersistentClient(path=persist_dir)
    collection = client.get_or_create_collection(name=collection_name)

    ids = [doc["id"] for doc in docs]
    documents = [doc["content"] for doc in docs]
    metadatas = [doc["metadata"] for doc in docs]

    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
    )

    return collection

In [41]:
import os

collection_path = os.getenv('CHROMA_COLLECTION_PATH', os.path.join(os.getcwd(),"collections"))

os.makedirs(collection_path,exist_ok=True)

collection = store_in_chroma(docs, collection_name = "duckdb_docs", persist_dir = collection_path)

In [42]:
import chromadb
client = chromadb.PersistentClient(collection_path)

In [43]:
client.list_collections()

[Collection(name=duckdb_docs)]

In [44]:
collection.count()

417

In [45]:
prompt = """
I found a CSV located in https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv. 
I want to read it, filter out only bad records (i.e. defaulters) and write them
out to a parquet file in my laptop at /Users/ss/duckdb-assistant/example_file.parquet.

"""
results = collection.query(query_texts = [prompt])

In [46]:
results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [47]:
results["distances"]

[[0.8144246935844421,
  0.8162638545036316,
  0.8616997599601746,
  0.869850754737854,
  0.8743537664413452,
  0.8918956518173218,
  0.8933833837509155,
  0.9238852262496948,
  0.9395614266395569,
  0.944223165512085]]

In [48]:
results["metadatas"][0][0]

{'source_path': '/Users/ss/duckdb-assistant/scripts/doc/docs/current/data/parquet/tips.md',
 'file_name': 'tips.md',
 'extension': '.md',
 'title': 'tips'}

In [49]:
import numpy as np

def find_elbow(ys):
    """
    Given a 1D array-like of y-values (e.g. inertia for k=1..K),
    return the index (0-based) of the elbow point.
    Assumes x is evenly spaced (1,2,3,...).
    """
    ys = np.asarray(ys, dtype=float)
    x = np.arange(len(ys))

    # First and second derivatives (central differences)
    # dy/dx
    dy = np.gradient(ys, x)
    # d²y/dx²
    d2y = np.gradient(dy, x)

    # Curvature approximation: |y''| / (1 + (y')²)^(3/2)
    curvature = np.abs(d2y) / (1 + dy**2)**1.5

    # Ignore endpoints if you want (optional)
    # curvature[0] = curvature[-1] = 0

    elbow_idx = int(np.argmax(curvature))
    return elbow_idx

In [50]:
find_elbow(results["distances"][0])

0

In [51]:
import numpy as np

def find_elbow_distance(ys):
    """
    Elbow as point farthest from the line connecting first and last points.
    Returns 0-based index.
    """
    ys = np.asarray(ys, dtype=float)
    x = np.arange(len(ys))

    # Line from (x0, y0) to (xn, yn)
    x0, y0 = x[0], ys[0]
    xn, yn = x[-1], ys[-1]

    # Vector from start to end
    vx, vy = xn - x0, yn - y0
    norm2 = vx**2 + vy**2

    # Distance of each point to the line (proportional, we don't need exact scale)
    distances = np.abs(vx * (ys - y0) - vy * (x - x0)) / np.sqrt(norm2)

    elbow_idx = int(np.argmax(distances))
    return elbow_idx

In [52]:
find_elbow_distance(results["distances"][0])

2

In [53]:
results["distances"][0][:find_elbow_distance(results["distances"][0])]

[0.8144246935844421, 0.8162638545036316]

In [54]:
results = collection.query(query_texts = [prompt], where_document={"$contains": "csv"} # query records with the search string in the records' document
)

In [55]:
results["distances"][0][:find_elbow_distance(results["distances"][0])]

[0.8743537664413452,
 0.8918956518173218,
 0.9395614266395569,
 0.944223165512085,
 1.0106737613677979,
 1.0172889232635498]

In [56]:
results["metadatas"][0][2]

{'source_path': '/Users/ss/duckdb-assistant/scripts/doc/docs/current/data/partitioning/partitioned_writes.md',
 'file_name': 'partitioned_writes.md',
 'title': 'partitioned_writes',
 'extension': '.md'}